# 6 — Fine-tuning a Published Pretrained EEG Encoder (CBraMod)

**Colab port of the EEGDash tutorial**
[`plot_73_finetune_pretrained_model`](https://eegdash.org/generated/auto_examples/tutorials/70_transfer_foundation/plot_73_finetune_pretrained_model.html)

A real checkpoint, downloaded from the Hub, adapted to a task it never saw. **CBraMod** was
pretrained on the TUH EEG corpus; here it decodes **eyes open vs eyes closed** from the HBN
R5 mini release — a different cohort, different hardware, different task.

The experiment is a three-way comparison, run identically under subject-grouped
cross-validation:

| Regime | Encoder | Head | What it tests |
|---|---|---|---|
| `scratch` | random init, trains | trains | does pretraining help at all? |
| `linear probe` | pretrained, **frozen** | trains | are the features already linearly separable? |
| `fine-tune` | pretrained, trains | trains | does adapting the encoder add more? |

Every participant is held out exactly once, so each regime ends with one balanced accuracy
per participant — paired scores, which is what makes the Wilcoxon test at the end legitimate.

**Runtime:** GPU recommended (Runtime → Change runtime type → T4). Expect ~20–40 min:
6 folds × 3 regimes = 18 training runs, plus the initial download.

## 0 · Install

`eegdash` for the HBN data, `braindecode` for `CBraMod` and its `from_pretrained`
loader. The restart afterwards is expected.

In [ ]:
%pip install -q eegdash braindecode

import IPython
print("Install finished — restarting the runtime.")
print("This 'crash' notice is expected. Continue at Section 1 below.")
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import copy
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from braindecode.models import CBraMod
from braindecode.preprocessing import (
    Preprocessor,
    create_windows_from_events,
    preprocess,
)
from scipy.stats import wilcoxon
from sklearn.metrics import balanced_accuracy_score

from eegdash import EEGChallengeDataset
from eegdash.const import SUBJECT_MINI_RELEASE_MAP

import mne
mne.set_log_level("ERROR")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} · device {DEVICE}")
if DEVICE == "cpu":
    print("CPU detected — 18 training runs will be slow. Runtime → Change runtime type → T4.")

## 1 · Configuration

Three choices fix everything downstream.

**Channels.** Only three posterior electrodes (`E70`, `E75`, `E83`). Eyes-closed alpha is an
occipital phenomenon; three channels keep the input small and the demonstration honest —
if it works, it works on genuine occipital alpha rather than on frontal EOG artifacts from
the eyes themselves.

**Sampling rate.** 200 Hz is not negotiable: CBraMod tokenises EEG into one-second patches
of exactly 200 samples. A 2-second window is therefore 2 patches per channel.

**Cue windows.** Each cue is followed by a transition, so we skip it and take a
steady-state stretch: 5–19 s after an eyes-open cue, 15–29 s after an eyes-closed cue. The
offsets differ per cue because the protocol's blocks do.

In [ ]:
CACHE_DIR = Path(os.environ.get("EEGDASH_CACHE_DIR", "~/.eegdash_cache")).expanduser()
CHANNELS = ["E70", "E75", "E83"]  # three posterior (occipital) channels
SFREQ = 200  # CBraMod expects 200 Hz: one-second patches of 200 samples
# Steady-state window (start, stop) in seconds after each cue, as in the eyes-open/closed tutorial
CUE_WINDOW = {"instructed_toOpenEyes": (5, 19), "instructed_toCloseEyes": (15, 29)}

# Optional: persist the download across sessions.
# from google.colab import drive
# drive.mount("/content/drive")
# CACHE_DIR = Path("/content/drive/MyDrive/eegdash_cache")

## 2 · Load and preprocess the HBN resting-state recordings

Two recordings are excluded by name: their posterior channels are flat or saturated, which
would silently poison both the per-channel standardisation and the folds they land in.
Exclusions like this belong at the top of a script where a reader can see them, not buried
in a filter.

The preprocessing chain is short because CBraMod expects raw-ish input:

- pick the three channels, resample to 200 Hz
- **per-channel z-scoring within each recording** — uses no labels, so it leaks nothing, and
  it removes the amplitude differences between participants that would otherwise dominate
- drop cues too close to the end of the recording to supply a full window

`drop_late_cues` needs `apply_on_array=False` because it operates on the `Raw` object's
annotations, not on the data array.

In [ ]:
# Load the resting-state recordings of the HBN R5 mini release, except two with
# flat or saturated posterior channels
BAD_RECORDINGS = ["NDARAP785CTE", "NDARCA740UC8"]
subjects = [
    s for s in sorted(SUBJECT_MINI_RELEASE_MAP["R5"]) if s not in BAD_RECORDINGS
]
dataset = EEGChallengeDataset(
    release="R5", mini=True, task="RestingState", subject=subjects, cache_dir=CACHE_DIR
)
print(f"{len(subjects)} participants")

In [ ]:
def drop_late_cues(raw):
    """The last eyes-open cue comes ~5 s before the end and cannot supply a full window."""
    fits = (
        raw.annotations.onset + max(stop for _, stop in CUE_WINDOW.values())
        < raw.times[-1]
    )
    return raw.set_annotations(raw.annotations[fits])


preprocess(
    dataset,
    [
        Preprocessor("pick", picks=CHANNELS),
        Preprocessor("resample", sfreq=SFREQ),
        # standardize each channel of each recording with its own mean and std (uses no labels)
        Preprocessor(
            lambda x: (x - x.mean(axis=1, keepdims=True)) / x.std(axis=1, keepdims=True)
        ),
        Preprocessor(drop_late_cues, apply_on_array=False),
    ],
)
print("Preprocessing done.")

### Cut the windows

Non-overlapping 2-second windows inside each steady-state stretch (`stride == size`).
Overlapping windows would put near-duplicate samples on both sides of a split and inflate
every score.

In [ ]:
# Cut 2 s windows: label 0 = eyes open, 1 = eyes closed
windows = create_windows_from_events(
    dataset,
    mapping={"instructed_toOpenEyes": 0, "instructed_toCloseEyes": 1},
    trial_start_offset_samples={
        cue: start * SFREQ for cue, (start, _) in CUE_WINDOW.items()
    },
    trial_stop_offset_samples={
        cue: stop * SFREQ for cue, (_, stop) in CUE_WINDOW.items()
    },
    window_size_samples=2 * SFREQ,
    window_stride_samples=2 * SFREQ,
)

## 3 · Arrays and subject-grouped folds

**Participants determine the folds, not windows.** This is the single most important line in
the notebook. Windows from one recording are highly correlated; a random window split would
put a participant on both sides and report an accuracy that measures nothing but
recognising that participant.

Six folds of three participants each. Within every fold the remaining participants split
again: **12 train**, **3 validate** (they choose which epoch's weights to keep), **3 test**
(scored once, never looked at during training). Three disjoint roles, which is what keeps
the final number clean.

In [ ]:
# Arrays for PyTorch: X = (windows, channels, samples), y = label, subject = participant of each window
X = np.stack([x for x, _, _ in windows]).astype("float32")
metadata = windows.get_metadata()
y = metadata["target"].to_numpy()
subject = metadata["subject"].to_numpy()
print(f"X {X.shape} | classes {np.bincount(y)} | participants {len(subjects)}")

# Six folds of three participants each; every participant is tested exactly once
folds = np.array_split(np.array(subjects), 6)

## 4 · Download the published checkpoint

`revision` pins the exact commit of the weights. Without it, the Hub repo could move under
you and a future re-run would quietly be a different experiment.

`return_encoder_output=True` makes the model emit patch features rather than class scores,
so we can attach our own two-class head.

In [ ]:
# Download the published checkpoint (about 20 MB); the revision pins the exact weights
pretrained = CBraMod.from_pretrained(
    "braindecode/cbramod-pretrained",
    revision="584cdc415913739a05d84bf0c1cb3db397764507",
    return_encoder_output=True,  # return the patch features instead of class scores
    n_chans=len(CHANNELS),
    n_times=2 * SFREQ,
    sfreq=SFREQ,
)
print("Checkpoint loaded.")

## 5 · Model factory and training loop

**Where the 1200 comes from:** 3 channels × 2 one-second patches × 200 features per patch.
That flattened vector feeds a single `Linear(1200, 2)`.

Three details that decide whether the comparison is fair:

- `copy.deepcopy(pretrained)` — each fold starts from the *same* untouched weights. Reusing
  one object would let fold 2 inherit fold 1's training.
- `requires_grad_(False)` for the linear probe, **plus** `model[0].eval()` inside the loop.
  Freezing parameters does not disable dropout; without the second line the "frozen"
  encoder still injects noise and the probe is handicapped.
- Different learning rates: `1e-3` for the probe (a fresh linear layer wants a large step),
  `1e-4` when the encoder trains (pretrained weights want small nudges, or you erase what
  you came for).

Checkpoint selection is by **validation** balanced accuracy — the test participants are
untouched until the score is read.

In [ ]:
REGIMES = ["scratch", "linear probe", "fine-tune"]


def make_model(regime):
    """CBraMod encoder + linear head. 3 channels x 2 patches x 200 features = 1200 inputs."""
    if regime == "scratch":
        encoder = CBraMod(
            n_chans=len(CHANNELS),
            n_times=2 * SFREQ,
            sfreq=SFREQ,
            return_encoder_output=True,
        )
    else:
        encoder = copy.deepcopy(pretrained)
    if regime == "linear probe":
        encoder.requires_grad_(False)  # freeze the encoder: only the head learns
    return torch.nn.Sequential(encoder, torch.nn.Flatten(), torch.nn.Linear(1200, 2))


def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.from_numpy(X)).argmax(1).numpy()


def fit(model, train, valid, lr, frozen, n_epochs=6, batch_size=16):
    """Train with AdamW; return the model with the weights of the best validation epoch."""
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    best_score, best_state = -1, None
    for epoch in range(n_epochs):
        model.train()
        if frozen:
            model[0].eval()  # frozen encoder: also switch off its dropout
        for batch in torch.randperm(int(train.sum())).split(
            batch_size
        ):  # shuffled mini-batches
            idx = np.flatnonzero(train)[batch.numpy()]
            loss = torch.nn.functional.cross_entropy(
                model(torch.from_numpy(X[idx])), torch.from_numpy(y[idx])
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        score = balanced_accuracy_score(y[valid], predict(model, X[valid]))
        if score > best_score:
            best_score, best_state = score, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model

## 6 · Cross-validated comparison

The long cell. 6 folds × 3 regimes = 18 training runs, each printing a line when its fold
finishes so you can watch progress.

Scores are recorded **per held-out participant**, not per fold — that is what gives the
statistics below their sample size and their pairing.

In [ ]:
torch.manual_seed(0)
scores = {
    regime: [] for regime in REGIMES
}  # one balanced accuracy per held-out participant
for test_subjects in folds:
    others = [s for s in subjects if s not in test_subjects]
    train = np.isin(subject, others[:-3])  # 12 participants train
    valid = np.isin(subject, others[-3:])  # 3 participants select the epoch
    test = np.isin(subject, test_subjects)  # 3 participants are scored once
    for regime in REGIMES:
        frozen = regime == "linear probe"
        model = fit(
            make_model(regime), train, valid, lr=1e-3 if frozen else 1e-4, frozen=frozen
        )
        pred = predict(model, X[test])
        for s in test_subjects:
            m = subject[test] == s
            scores[regime].append(balanced_accuracy_score(y[test][m], pred[m]))
    n = len(test_subjects)
    print(
        f"held out {test_subjects.tolist()}: "
        + " | ".join(f"{r} {np.mean(scores[r][-n:]):.2f}" for r in REGIMES)
    )

## 7 · Statistics

Two questions, two tests.

**Is each regime above chance?** A one-sided Wilcoxon signed-rank test on `score − 0.5`.
Non-parametric, because 18 participant-level accuracies are neither normal nor plentiful.

**Does fine-tuning beat training from scratch?** A *paired* Wilcoxon over the same
participants. Pairing is what removes between-participant variability — some people simply
have cleaner alpha — and it is only available because every participant was held out under
all three regimes.

In [ ]:
for regime in REGIMES:
    s = np.array(scores[regime])
    p = wilcoxon(s - 0.5, alternative="greater").pvalue
    print(
        f"{regime:13s} mean {s.mean():.3f} | above chance in {(s > 0.5).sum()}/{len(s)} | Wilcoxon p = {p:.1e}"
    )
p = wilcoxon(scores["fine-tune"], scores["scratch"], alternative="greater").pvalue
print(f"fine-tune > scratch (paired over participants): p = {p:.1e}")

## 8 · Per-participant results

Bars are regime means; each dot is one held-out participant, dots in the same horizontal
position across bars being the same person. Read the **spread**, not just the bar: a high
mean carried by a few participants is a very different claim from a consistent gain.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
jitter = np.linspace(
    -0.2, 0.2, len(subjects)
)  # spread the dots; scores stay in participant order
for i, regime in enumerate(REGIMES):
    ax.bar(i, np.mean(scores[regime]), color="lightgray")
    ax.scatter(i + jitter, scores[regime], s=18, color="k", zorder=3)
ax.axhline(0.5, ls="--", color="gray")  # chance level
ax.set(
    xticks=range(len(REGIMES)),
    xticklabels=REGIMES,
    ylim=(0, 1),
    ylabel="Balanced accuracy per held-out participant",
)
plt.show()

## What to take away

- **Pretrained weights beat random initialisation**, on a cohort and a task the checkpoint
  never saw — the whole premise of transfer, demonstrated rather than asserted.
- **The frozen encoder already separates eye states** for most participants. That is the
  linear probe's job: it says the information is present in the representation, not
  manufactured by fine-tuning.
- **Fine-tuning is strongest**, but the gap over the probe is the part most worth
  scrutinising on your own data.

**Honest limits of this demo.** One seed, six epochs, three channels, eighteen participants.
Before attributing an improvement to the encoder itself you would want repeated seeds,
prespecified longer training budgets, and per-participant confusion matrices — an
eyes-open/closed asymmetry can produce a respectable balanced accuracy for reasons that have
nothing to do with the representation.

### Things to try

- Raise `n_epochs` and re-run: does `scratch` close the gap given time?
- Add more channels — does the advantage of pretraining grow or shrink?
- Loop over 3–5 seeds and plot the variance across them.
- Swap in another checkpoint from `braindecode.models` and compare under the same folds.